# 03 — Modelos Baseline y Evaluación
**CRISP-DM: Modelado + Evaluación** · Etapa 1: *Baseline (≥3 algoritmos) + tabla comparativa + interpretabilidad*

Requiere `pip install -r requirements.txt` (scikit-learn, xgboost, shap).
Usa `data/processed/dataset_modelado.csv` generado en el notebook 02.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
import pandas as pd
from prediccion_precios import features as ft, evaluation as ev, config
from prediccion_precios import models_baseline as mb, interpretability as it
pd.set_option("display.width",160); pd.set_option("display.max_columns",60)

In [ ]:
data = pd.read_csv(config.DATASET_MODELADO, parse_dates=[config.COL_FECHA])
X, y = ft.construir_matriz_modelado(data)
X = X.astype(float)
X_train, X_test, y_train, y_test = ev.split_temporal(X, y)
print("X", X.shape, "| train", len(X_train), "| test", len(X_test))

## 1. Entrenar los 3 baselines (Regresión Lineal, Random Forest, XGBoost)

In [ ]:
modelos = mb.entrenar_todos(X_train, y_train)
list(modelos.keys())

## 2. Tabla comparativa de métricas (MAE / RMSE / MAPE / R²)

In [ ]:
resultados = {n: ev.calcular_metricas(y_test, m.predict(X_test)) for n, m in modelos.items()}
tabla = ev.tabla_comparativa(resultados)
print("Meta objetivo: MAPE <", config.META_MAPE_OBJETIVO*100, "%")
tabla

## 3. Validación cruzada temporal (ventana expansiva)
Se pasa la *fábrica* del modelo (`mb.FABRICAS[nombre]`) para re-crearlo en cada fold.

In [ ]:
for nombre, fabrica in mb.FABRICAS.items():
    cv = ev.validacion_cruzada_temporal(fabrica, X, y, n_splits=config.CV_SPLITS)
    print(nombre, "-> MAPE %:", cv["MAPE_%"], "| R2:", cv["R2"])

## 4. Interpretabilidad — importancia de variables + SHAP

In [ ]:
mejor = modelos["xgboost"]
display(it.importancia_variables(mejor, X.columns).head(15))
it.graficar_importancia(mejor, X.columns)            # -> reports/figures/importancia_variables.png
it.explicar_shap(mejor, X_test)                       # -> reports/figures/shap_summary.png

## 5. Guardar los modelos entrenados

In [ ]:
for nombre, modelo in modelos.items():
    mb.guardar_modelo(modelo, nombre)
print("Modelos guardados en", config.MODELS_DIR)

### Conclusiones de Etapa 1
> Indicar el mejor baseline, su MAPE frente a la meta (<15%), las variables más
> influyentes según SHAP y los próximos pasos hacia la Etapa 2 (optimización de
> hiperparámetros, LSTM/redes neuronales y ensemble).